In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/100_Unique_QA_Dataset.csv')

In [3]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
df.isnull().value_counts()

,,count
question,answer,
False,False,90


In [5]:
df.isna().value_counts()

,,count
question,answer,
False,False,90


In [6]:
def tokenize(text):
  text = text.lower()
  text = text.replace('?', ' ')
  text = text.replace("'", "")
  return text.split()

In [7]:
tokenize('what is the capital of france?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [8]:
vocab = {'<UNK>':0}

In [9]:
vocab = {}

def build_vocab(row):
    question_tokens = tokenize(row['question'])
    answer_tokens = tokenize(row['answer'])

    print("Q:", question_tokens)
    print("A:", answer_tokens)

    merged = question_tokens + answer_tokens

    for word in merged:
        if word not in vocab:
            vocab[word] = len(vocab)

In [10]:
df.apply(build_vocab, axis=1)

Q: ['what', 'is', 'the', 'capital', 'of', 'france']
A: ['paris']
Q: ['what', 'is', 'the', 'capital', 'of', 'germany']
A: ['berlin']
Q: ['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']
A: ['harper-lee']
Q: ['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system']
A: ['jupiter']
Q: ['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius']
A: ['100']
Q: ['who', 'painted', 'the', 'mona', 'lisa']
A: ['leonardo-da-vinci']
Q: ['what', 'is', 'the', 'square', 'root', 'of', '64']
A: ['8']
Q: ['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold']
A: ['au']
Q: ['which', 'year', 'did', 'world', 'war', 'ii', 'end']
A: ['1945']
Q: ['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world']
A: ['nile']
Q: ['what', 'is', 'the', 'capital', 'of', 'japan']
A: ['tokyo']
Q: ['who', 'developed', 'the', 'theory', 'of', 'relativity']
A: ['albert-einstein']
Q: ['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit']
A: ['32']
Q: ['which', 'planet',

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [11]:
vocab

{'what': 0,
 'is': 1,
 'the': 2,
 'capital': 3,
 'of': 4,
 'france': 5,
 'paris': 6,
 'germany': 7,
 'berlin': 8,
 'who': 9,
 'wrote': 10,
 'to': 11,
 'kill': 12,
 'a': 13,
 'mockingbird': 14,
 'harper-lee': 15,
 'largest': 16,
 'planet': 17,
 'in': 18,
 'our': 19,
 'solar': 20,
 'system': 21,
 'jupiter': 22,
 'boiling': 23,
 'point': 24,
 'water': 25,
 'celsius': 26,
 '100': 27,
 'painted': 28,
 'mona': 29,
 'lisa': 30,
 'leonardo-da-vinci': 31,
 'square': 32,
 'root': 33,
 '64': 34,
 '8': 35,
 'chemical': 36,
 'symbol': 37,
 'for': 38,
 'gold': 39,
 'au': 40,
 'which': 41,
 'year': 42,
 'did': 43,
 'world': 44,
 'war': 45,
 'ii': 46,
 'end': 47,
 '1945': 48,
 'longest': 49,
 'river': 50,
 'nile': 51,
 'japan': 52,
 'tokyo': 53,
 'developed': 54,
 'theory': 55,
 'relativity': 56,
 'albert-einstein': 57,
 'freezing': 58,
 'fahrenheit': 59,
 '32': 60,
 'known': 61,
 'as': 62,
 'red': 63,
 'mars': 64,
 'author': 65,
 '1984': 66,
 'george-orwell': 67,
 'currency': 68,
 'united': 69,
 'kin

In [12]:
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [13]:
text_to_indices('what is the capital of france?', vocab)

[0, 1, 2, 3, 4, 5]

In [14]:
vocab

{'what': 0,
 'is': 1,
 'the': 2,
 'capital': 3,
 'of': 4,
 'france': 5,
 'paris': 6,
 'germany': 7,
 'berlin': 8,
 'who': 9,
 'wrote': 10,
 'to': 11,
 'kill': 12,
 'a': 13,
 'mockingbird': 14,
 'harper-lee': 15,
 'largest': 16,
 'planet': 17,
 'in': 18,
 'our': 19,
 'solar': 20,
 'system': 21,
 'jupiter': 22,
 'boiling': 23,
 'point': 24,
 'water': 25,
 'celsius': 26,
 '100': 27,
 'painted': 28,
 'mona': 29,
 'lisa': 30,
 'leonardo-da-vinci': 31,
 'square': 32,
 'root': 33,
 '64': 34,
 '8': 35,
 'chemical': 36,
 'symbol': 37,
 'for': 38,
 'gold': 39,
 'au': 40,
 'which': 41,
 'year': 42,
 'did': 43,
 'world': 44,
 'war': 45,
 'ii': 46,
 'end': 47,
 '1945': 48,
 'longest': 49,
 'river': 50,
 'nile': 51,
 'japan': 52,
 'tokyo': 53,
 'developed': 54,
 'theory': 55,
 'relativity': 56,
 'albert-einstein': 57,
 'freezing': 58,
 'fahrenheit': 59,
 '32': 60,
 'known': 61,
 'as': 62,
 'red': 63,
 'mars': 64,
 'author': 65,
 '1984': 66,
 'george-orwell': 67,
 'currency': 68,
 'united': 69,
 'kin

In [15]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab
  def __len__(self):
    return self.df.shape[0]
  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [16]:
dataset = QADataset(df, vocab)

In [17]:
dataloader = DataLoader(dataset, batch_size = 1, shuffle = True)

In [18]:
for q,a in dataloader:
  print(q,a)

tensor([[ 77,  78, 149, 150,  13, 151, 152]]) tensor([[153]])
tensor([[ 41,  17, 117,   2, 185, 186]]) tensor([[187]])
tensor([[  0,   1,   2,   3,   4, 108]]) tensor([[316]])
tensor([[  9,  10, 156, 157, 158]]) tensor([[159]])
tensor([[ 41, 116, 117,   2, 118,  93, 119]]) tensor([[120]])
tensor([[ 41, 215, 117, 216, 217,  18,  13, 218,  42]]) tensor([[219]])
tensor([[  0,   1,   2,  36, 132,   4,  25]]) tensor([[133]])
tensor([[ 41, 136,   1, 137,  38, 138]]) tensor([[52]])
tensor([[0, 1, 2, 3, 4, 5]]) tensor([[6]])
tensor([[  0,   1,   2,   3,   4, 134]]) tensor([[135]])
tensor([[  0,   1,   2, 179, 180, 181, 182]]) tensor([[183]])
tensor([[ 41, 173,   1,  61,  38, 174, 175,  11, 176, 177]]) tensor([[178]])
tensor([[ 77,  78, 287,  80,  18,  13, 288]]) tensor([[84]])
tensor([[  0,   1,   2,   3,   4, 285]]) tensor([[286]])
tensor([[ 77,  78, 194,  80,  18,   2, 195, 196, 197]]) tensor([[198]])
tensor([[  9, 139,   2, 140, 269,  92, 270,   4,   2, 271]]) tensor([[272]])
tensor([[  0, 

In [19]:
import torch.nn as nn
class nnRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 50)
    self.rnn = nn.RNN(50, 64, batch_first=True) # Add batch_first=True
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, x):
    x = self.embedding(x) # x: (batch_size, seq_len_q, embedding_dim)

    # output: (batch_size, seq_len_q, hidden_size)
    # h: (num_layers * num_directions, batch_size, hidden_size)
    # For a simple RNN, h is the final hidden state for each element in the batch.
    _, h = self.rnn(x) # We only need the last hidden state h

    # For a single-layer, uni-directional RNN, h will have shape (1, batch_size, hidden_size).
    # We need to squeeze out the first dimension (num_layers) to get (batch_size, hidden_size)
    # to feed into the linear layer.
    x = self.fc(h.squeeze(0)) # x: (batch_size, vocab_size)
    return x


In [20]:
learning_rate = 0.01
epochs = 20

In [21]:
model = nnRNN(len(vocab))

In [22]:
criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [23]:
for epoch in range(epochs):
  total_loss = 0
  for q, a in dataloader:
    optimizer.zero_grad()
    output = model(q)
    # Squeeze the target 'a' to (batch_size,) as expected by CrossEntropyLoss
    loss = criteria(output, a.squeeze(1))
    loss.backward()
    optimizer.step()
    total_loss = total_loss + loss.item()
  print(f'epoch: {epoch+1}, loss: {total_loss:4f}')


epoch: 1, loss: 539.505683
epoch: 2, loss: 318.609594
epoch: 3, loss: 160.854129
epoch: 4, loss: 61.386023
epoch: 5, loss: 32.615368
epoch: 6, loss: 18.016381
epoch: 7, loss: 15.873919
epoch: 8, loss: 10.148080
epoch: 9, loss: 14.069743
epoch: 10, loss: 10.775402
epoch: 11, loss: 8.145693
epoch: 12, loss: 14.029299
epoch: 13, loss: 9.338275
epoch: 14, loss: 15.316695
epoch: 15, loss: 16.041831
epoch: 16, loss: 10.630182
epoch: 17, loss: 8.950038
epoch: 18, loss: 15.118300
epoch: 19, loss: 39.467816
epoch: 20, loss: 29.620271


In [24]:
total = 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in dataloader:
    y_pred = model(batch_features)
    _, predicted = torch.max(y_pred, 1)
    total += batch_labels.size(0) # shape[0] will also work the same.
    correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(accuracy)

0.9222222222222223


In [25]:
def predict(model, question, threshold = 0.5):
  numerical_question = text_to_indices(question, vocab)
  numerical_question = torch.tensor(numerical_question).unsqueeze(0)

  # Get raw logits from the model
  logits = model(numerical_question)

  # Apply softmax to get probabilities
  probs = torch.softmax(logits, dim=1)

  # Get the predicted class index and its probability
  value, item = torch.max(probs, dim=1)

  # Check against threshold
  if value < threshold:
    return "I don't know"

  # Find the word corresponding to the predicted index
  predicted_index = item.item() # Get the scalar index

  # Invert the vocab dictionary to map indices back to words
  idx_to_word = {idx: word for word, idx in vocab.items()}

  try:
    predicted_word = idx_to_word[predicted_index]
    return predicted_word
  except KeyError:
    return "Predicted index not found in vocab"


In [26]:
predict(model, "What is the boiling point of water in Celsius?")

'100'

In [27]:
output.shape

torch.Size([1, 323])

In [28]:
!pip install nltk

In [29]:
import nltk
import torch
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [30]:
document = """
Over centuries, the Bhagavad Gita has been considered animportant spiritual guidebook and has influenced many thought leaders inhuman history. The Gita is a conversational poem and is part of the greatIndian epic the Mahabharata, which is a story about the families of twobrothers who inherit a kingdom from their father. These cousins are knownas the ‘Pandavas’ and the ‘Kauravas’. The Kaurava brothers are 100 innumber and the Pandavas are 5. The Kauravas, not wanting to share thekingdom with their cousins, invite them to a game of dice. They usedubious means to defeat the Pandavas and annex their kingdom afterwinning the wager. As per the rules of the wager, the Pandavas completetheir stay outside the kingdom and come back to ask for their fair share ofthe kingdom so that they can rule with dignity. The Kauravas refuse tohonor the agreement and even deny the Pandavas a settlement of 5 villagesso that they can live in peace. As the Kauravas refuse to budge even aninch, the Pandavas have no choice but to declare war on them to get backtheir share of the kingdom. During this period, The Lord who hasincarnated as Krishna offers the cousins a choice between His army andHimself. But He makes it clear that He Himself would not be participatingin the war. The Kauravas choose His army as Krishna would not befighting, and the Pandavas are happy to just have Krishna on their side.Krishna takes on the role of charioteer to Arjuna’s chariot (Arjuna is thethird brother among the five Pandavas).Before the war starts, the two armies assemble on the battlefield,facing each other. At this point, Arjuna requests Krishna to place hischariot in the middle so that he can get a full view of the warring forces.Since this is a war between cousins, Arjuna sees his cousins, uncles,grandfathers, great grandfathers, nephews, friends, classmates etc. on bothsides. He is overcome with emotion and grief at the prospect of bloodshedamong his family and friends for the sake of a kingdom. He declares hisintention to renounce the war and let the Kauravas keep the kingdom, thus abdicating his responsibility as a prince and a warrior. It is at this stage thatKrishna teaches him the Gita which is structured as a series of questionsfrom Arjuna and answers from Krishna. Krishna extols the virtue ofperforming one’s duties regardless of the outcome, and not avoidingprescribed duties which may be difficult and unpleasant. As Krishna startsto talk to Arjuna on the virtues of doing one’s work (in this case Arjunaneeding to fight a rightful war), Arjuna asks a series of questions on thebigger issues of life, individual souls, the Lord Almighty, the universe, thecircle of life etc., for which Krishna provides clear, unambiguous answers.He finally convinces Arjuna that his fears were unfounded and that heshould fight to free the Kingdom from the Kauravas and provide a just andcompassionate administration to his citizens.At first a battleground is hardly the place for someone to bepreaching philosophy. However, many of the questions that we have in lifeare about the choices we have to make, especially in challenging times. It isinteresting to note that Sanskrit verses lend themselves to multiplemeanings and in one interpretation, the battleground in the Gita iscompared to the human body (and mind), and the battle between thePandavas and the Kauravas is compared to the constant strife between goodand evil thoughts that we encounter daily. The Gita is indeed a teaching forall of us, with Arjuna acting as an example of an individual at crossroads,desperately looking for guidance and support.The Gita is one of the most widely read and commented uponspiritual works in human history. There are several excellent books on it invarious languages. Given that the original work is in Sanskrit, most of thescholarly commentaries have been in Sanskrit or other Indian languageswhich are closer to Sanskrit. To read and comprehend many of them, onewould need some training in Indian spiritual studies, as many of themliberally use Sanskrit words albeit transliterated in English. It was felt thatthere is a need to present the key concepts of The Gita in plain English foreveryday folk, minimizing the use of Sanskrit words. These observationsled to the development of this manuscript. While there are many learned commentaries on the Gita written overthe centuries, this author has been greatly influenced by the authoritativecommentary and lucid explanations of difficult concepts by SriMadhwacharya, the 12th century ascetic who propounded the philosophy ofdualistic theism (concept of difference between the almighty Lord andindividual souls at all times and places). Sri Madhwacharya’s commentaryhas been further elaborated and summarized by later day savants such asSri Padmanabha Teertha (13th Century), Sri Jaya Teertha (13th Century) SriVadiraja Teertha (16th Century), Sri Raghavendra Teertha (17th century)and others.The Gita consists of 18 chapters with a total of 7011 verses inSanskrit. The Gita is part of the great Hindu epic The Mahabharata andappears as Chapters 25 to 42, in the Book of Bheeshma in theMahabharata. Most of the verses appear in the anushtap meter, whereeach verse has a total of 32 letters with 8 words in each of the fourquadrants. Some verses in the 15th chapter appear in trishtup meter with 44words. As per Sri Raghavendra Teertha [GV], the 18 chapters can bebroadly classified into 3 sections with 6 chapters in each section. The firstsection with chapters 1-6 (280 verses) broadly outlines how one can obtaindivine, spiritual knowledge (jnana upaya), the second section with chapters7-12 (209 verses) focuses on devotion and divine, spiritual knowledge(vijnana) that leads one to liberation, and the third section consisting ofchapters 13-18 (212 verses) elaborates on the concepts in the first twosections. It is also believed that the Gita conveys at least 10 differentmeanings, all of which are consistent with one another. An illustration ofmultiple meanings in the Gita will appear in Chapter 2. It is also said thatthe Gita has 3 major interpretations – historical, psychological, and spiritual. Numbers and numerology play an incredibly significant role inIndian theology. In fact, the study of knowledge is termed ‘Sankhya Yoga’,with Sankhya meaning ‘arithmetic’. The world around us is based onnumbers. We are constantly measuring, managing, quantifying all ouractivities with the aid of numbers. We wake up at a certain time, wework/study for certain hours, we travel a certain distance, we consumesome amount of food, we transact with certain currency, we performactivities at a certain time, on a certain day, in a certain month, in a certainyear etc. And all these numbers and calculations are based on the positionand movement of celestial objects such as the Sun, Moon, stars etc."""

In [31]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [32]:
tokens = word_tokenize(document.lower())
tokens

['over',
 'centuries',
 ',',
 'the',
 'bhagavad',
 'gita',
 'has',
 'been',
 'considered',
 'animportant',
 'spiritual',
 'guidebook',
 'and',
 'has',
 'influenced',
 'many',
 'thought',
 'leaders',
 'inhuman',
 'history',
 '.',
 'the',
 'gita',
 'is',
 'a',
 'conversational',
 'poem',
 'and',
 'is',
 'part',
 'of',
 'the',
 'greatindian',
 'epic',
 'the',
 'mahabharata',
 ',',
 'which',
 'is',
 'a',
 'story',
 'about',
 'the',
 'families',
 'of',
 'twobrothers',
 'who',
 'inherit',
 'a',
 'kingdom',
 'from',
 'their',
 'father',
 '.',
 'these',
 'cousins',
 'are',
 'knownas',
 'the',
 '‘',
 'pandavas',
 '’',
 'and',
 'the',
 '‘',
 'kauravas',
 '’',
 '.',
 'the',
 'kaurava',
 'brothers',
 'are',
 '100',
 'innumber',
 'and',
 'the',
 'pandavas',
 'are',
 '5.',
 'the',
 'kauravas',
 ',',
 'not',
 'wanting',
 'to',
 'share',
 'thekingdom',
 'with',
 'their',
 'cousins',
 ',',
 'invite',
 'them',
 'to',
 'a',
 'game',
 'of',
 'dice',
 '.',
 'they',
 'usedubious',
 'means',
 'to',
 'defeat'

In [33]:
vocab = {'<unk>':0}
Counter(tokens).keys()

dict_keys(['over', 'centuries', ',', 'the', 'bhagavad', 'gita', 'has', 'been', 'considered', 'animportant', 'spiritual', 'guidebook', 'and', 'influenced', 'many', 'thought', 'leaders', 'inhuman', 'history', '.', 'is', 'a', 'conversational', 'poem', 'part', 'of', 'greatindian', 'epic', 'mahabharata', 'which', 'story', 'about', 'families', 'twobrothers', 'who', 'inherit', 'kingdom', 'from', 'their', 'father', 'these', 'cousins', 'are', 'knownas', '‘', 'pandavas', '’', 'kauravas', 'kaurava', 'brothers', '100', 'innumber', '5.', 'not', 'wanting', 'to', 'share', 'thekingdom', 'with', 'invite', 'them', 'game', 'dice', 'they', 'usedubious', 'means', 'defeat', 'annex', 'afterwinning', 'wager', 'as', 'per', 'rules', 'completetheir', 'stay', 'outside', 'come', 'back', 'ask', 'for', 'fair', 'ofthe', 'so', 'that', 'can', 'rule', 'dignity', 'refuse', 'tohonor', 'agreement', 'even', 'deny', 'settlement', '5', 'villagesso', 'live', 'in', 'peace', 'budge', 'aninch', 'have', 'no', 'choice', 'but', 'dec

In [34]:
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)
vocab

{'<unk>': 0,
 'over': 1,
 'centuries': 2,
 ',': 3,
 'the': 4,
 'bhagavad': 5,
 'gita': 6,
 'has': 7,
 'been': 8,
 'considered': 9,
 'animportant': 10,
 'spiritual': 11,
 'guidebook': 12,
 'and': 13,
 'influenced': 14,
 'many': 15,
 'thought': 16,
 'leaders': 17,
 'inhuman': 18,
 'history': 19,
 '.': 20,
 'is': 21,
 'a': 22,
 'conversational': 23,
 'poem': 24,
 'part': 25,
 'of': 26,
 'greatindian': 27,
 'epic': 28,
 'mahabharata': 29,
 'which': 30,
 'story': 31,
 'about': 32,
 'families': 33,
 'twobrothers': 34,
 'who': 35,
 'inherit': 36,
 'kingdom': 37,
 'from': 38,
 'their': 39,
 'father': 40,
 'these': 41,
 'cousins': 42,
 'are': 43,
 'knownas': 44,
 '‘': 45,
 'pandavas': 46,
 '’': 47,
 'kauravas': 48,
 'kaurava': 49,
 'brothers': 50,
 '100': 51,
 'innumber': 52,
 '5.': 53,
 'not': 54,
 'wanting': 55,
 'to': 56,
 'share': 57,
 'thekingdom': 58,
 'with': 59,
 'invite': 60,
 'them': 61,
 'game': 62,
 'dice': 63,
 'they': 64,
 'usedubious': 65,
 'means': 66,
 'defeat': 67,
 'annex': 6

In [35]:
len(vocab)

509

In [36]:
input_sentences = document.split("\n")

In [37]:
def text_to_indices(sentence, vocab):
  indexed_text = []
  for token in sentence:
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<unk>'])
  return indexed_text

In [38]:
input_sentences

['',
 'Over centuries, the Bhagavad Gita has been considered animportant spiritual guidebook and has influenced many thought leaders inhuman history. The Gita is a conversational poem and is part of the greatIndian epic the Mahabharata, which is a story about the families of twobrothers who inherit a kingdom from their father. These cousins are knownas the ‘Pandavas’ and the ‘Kauravas’. The Kaurava brothers are 100 innumber and the Pandavas are 5. The Kauravas, not wanting to share thekingdom with their cousins, invite them to a game of dice. They usedubious means to defeat the Pandavas and annex their kingdom afterwinning the wager. As per the rules of the wager, the Pandavas completetheir stay outside the kingdom and come back to ask for their fair share ofthe kingdom so that they can rule with dignity. The Kauravas refuse tohonor the agreement and even deny the Pandavas a settlement of 5 villagesso that they can live in peace. As the Kauravas refuse to budge even aninch, the Pandava

In [39]:
input_numerical_sentences = []
for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [40]:
input_numerical_sentences

[[],
 [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  7,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  4,
  6,
  21,
  22,
  23,
  24,
  13,
  21,
  25,
  26,
  4,
  27,
  28,
  4,
  29,
  3,
  30,
  21,
  22,
  31,
  32,
  4,
  33,
  26,
  34,
  35,
  36,
  22,
  37,
  38,
  39,
  40,
  20,
  41,
  42,
  43,
  44,
  4,
  45,
  46,
  47,
  13,
  4,
  45,
  48,
  47,
  20,
  4,
  49,
  50,
  43,
  51,
  52,
  13,
  4,
  46,
  43,
  53,
  4,
  48,
  3,
  54,
  55,
  56,
  57,
  58,
  59,
  39,
  42,
  3,
  60,
  61,
  56,
  22,
  62,
  26,
  63,
  20,
  64,
  65,
  66,
  56,
  67,
  4,
  46,
  13,
  68,
  39,
  37,
  69,
  4,
  70,
  20,
  71,
  72,
  4,
  73,
  26,
  4,
  70,
  3,
  4,
  46,
  74,
  75,
  76,
  4,
  37,
  13,
  77,
  78,
  56,
  79,
  80,
  39,
  81,
  57,
  82,
  37,
  83,
  84,
  64,
  85,
  86,
  59,
  87,
  20,
  4,
  48,
  88,
  89,
  4,
  90,
  13,
  91,
  92,
  4,
  46,
  22,
  93,
  26,
  94,
  95,
  84,
  64,
  85,
  96,
  97,
  98,
  20,
  

In [41]:
len(input_numerical_sentences)

2

In [42]:
training_sequence = []
for sentence in input_numerical_sentences:
  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [43]:
len(training_sequence)

1236

In [44]:
training_sequence[1]

[1, 2, 3]

In [45]:
len_list = []
for sequence in training_sequence:
  len_list.append(len(sequence))
max(len_list)

1237

In [46]:
padded_training_sequence = []
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [47]:
len(padded_training_sequence[6])

1237

In [48]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype = torch.long)
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   1,  ..., 507,   3, 508],
        [  0,   1,   2,  ...,   3, 508, 172],
        [  1,   2,   3,  ..., 508, 172,  20]])

In [49]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [50]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   1,  ...,   3, 507,   3],
        [  0,   1,   2,  ..., 507,   3, 508],
        [  1,   2,   3,  ...,   3, 508, 172]])

In [51]:
y

tensor([  2,   3,   4,  ..., 508, 172,  20])

In [52]:
class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y
  def __len__(self):
    return self.X.shape[0]
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [53]:
dataset = CustomDataset(X, y)
dataset[0]

(tensor([0, 0, 0,  ..., 0, 0, 1]), tensor(2))

In [54]:
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [55]:
for input, output in dataloader:
  print(input, output)

tensor([[  0,   0,   0,  ...,   4, 380, 381],
        [  0,   0,   0,  ..., 343, 347,   3],
        [  0,   0,   0,  ...,  34,  35,  36],
        ...,
        [  0,   0,   0,  ..., 309, 310,  43],
        [  0,   0,   0,  ..., 483,  59,   4],
        [  0,   0,   0,  ..., 381,  97, 153]]) tensor([ 85,   4,  22,  56,  54, 221, 385,  42, 487,   4,  18, 294,   4,  20,
        325,  97, 112, 394,   4, 502,  69, 461,   4,  56, 224,  13, 467, 124,
         13, 311, 484, 414])
tensor([[  0,   0,   0,  ..., 353, 140, 354],
        [  0,   0,   0,  ..., 259,  84, 260],
        [  0,   0,   0,  ...,  45, 474,  47],
        ...,
        [  0,   0,   0,  ...,  13,  84, 238],
        [  0,   0,   0,  ...,  97,  22, 487],
        [  0,   0,   0,  ..., 158, 159,  97]]) tensor([ 26, 261,  20, 164,  97,  20,  31,  53,   3,  20,   3, 225,   6,   4,
         13, 410, 130, 357,  45, 136,  21,   4, 251,   6,   3,  47,  12,  43,
        202, 217, 499,   4])
tensor([[  0,   0,   0,  ..., 223,   3, 224],
    

In [56]:
import torch.nn as nn
class LSTMModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    # Fix: Embedding dimension should match LSTM input_size
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, x): # Fix: Added x as input argument
    embedded = self.embedding(x) # Fix: Use the passed x instead of global X
    output, (hidden, cell) = self.lstm(embedded)
    # hidden shape: (num_layers * num_directions, batch_size, hidden_size)
    # For a single layer, uni-directional LSTM, it's (1, batch_size, hidden_size)
    output = self.fc(hidden.squeeze(0)) # Squeeze the first dimension
    return output

In [57]:
model = LSTMModel(len(vocab))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LSTMModel(
  (embedding): Embedding(509, 100)
  (lstm): LSTM(100, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=509, bias=True)
)

In [58]:
epochs = 50
learning_rate = 0.001 # Reduced learning rate
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [59]:
for epoch in range(epochs):
  total_loss = 0
  for input_data, target_labels in dataloader: # Renamed 'output' to 'target_labels' for clarity
    input_data = input_data.to(device)
    target_labels = target_labels.to(device) # Move target_labels to device
    optimizer.zero_grad()
    model_outputs = model(input_data) # Renamed 'output' to 'model_outputs' for clarity
    loss = criterion(model_outputs, target_labels) # Fix: Use model_outputs and target_labels
    loss.backward()
    optimizer.step()
    total_loss = total_loss + loss.item()
  print(f"Epoch: {epoch + 1}, Loss: {total_loss: .4f}")

Epoch: 1, Loss:  241.4754
Epoch: 2, Loss:  226.2052
Epoch: 3, Loss:  207.6998
Epoch: 4, Loss:  199.7188
Epoch: 5, Loss:  192.7254
Epoch: 6, Loss:  185.9798
Epoch: 7, Loss:  178.7901
Epoch: 8, Loss:  171.4679
Epoch: 9, Loss:  164.6870
Epoch: 10, Loss:  157.6979
Epoch: 11, Loss:  150.7908
Epoch: 12, Loss:  143.9098
Epoch: 13, Loss:  137.5026
Epoch: 14, Loss:  131.0812
Epoch: 15, Loss:  124.5512
Epoch: 16, Loss:  118.3652
Epoch: 17, Loss:  112.3932
Epoch: 18, Loss:  106.4339
Epoch: 19, Loss:  100.9740
Epoch: 20, Loss:  95.6122
Epoch: 21, Loss:  90.3826
Epoch: 22, Loss:  85.5044
Epoch: 23, Loss:  80.4109
Epoch: 24, Loss:  75.9162
Epoch: 25, Loss:  71.4838
Epoch: 26, Loss:  67.0259
Epoch: 27, Loss:  63.0183
Epoch: 28, Loss:  59.2359
Epoch: 29, Loss:  55.4217
Epoch: 30, Loss:  52.0717
Epoch: 31, Loss:  48.8022
Epoch: 32, Loss:  45.7546
Epoch: 33, Loss:  42.7917
Epoch: 34, Loss:  40.0971
Epoch: 35, Loss:  37.7507
Epoch: 36, Loss:  35.4244
Epoch: 37, Loss:  33.1490
Epoch: 38, Loss:  31.2389
Ep

In [60]:
model.eval() # Set the model to evaluation mode
total = 0
correct = 0
with torch.no_grad():
  for input_data, target_labels in dataloader:
    input_data = input_data.to(device)
    target_labels = target_labels.to(device)
    model_outputs = model(input_data)
    _, predicted = torch.max(model_outputs, 1)
    total += target_labels.size(0)
    correct += (predicted == target_labels).sum().item()

accuracy = correct / total
print(f"Training Accuracy: {accuracy*100:.2f}%")
model.train() # Set the model back to training mode

Training Accuracy: 99.43%


LSTMModel(
  (embedding): Embedding(509, 100)
  (lstm): LSTM(100, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=509, bias=True)
)

In [61]:
def predict_next_word(model, input_sentence, vocab, max_len):
  model.eval() # Set the model to evaluation mode
  tokens = word_tokenize(input_sentence.lower()) # Tokenize and lowercase
  numerical_input = [vocab.get(token, vocab['<unk>']) for token in tokens]

  # Pad the input sequence
  if len(numerical_input) > max_len - 1: # -1 because the last element is the target
      numerical_input = numerical_input[-(max_len - 1):] # Take the last max_len-1 tokens
  padded_input = [0] * (max_len - 1 - len(numerical_input)) + numerical_input

  input_tensor = torch.tensor(padded_input, dtype=torch.long).unsqueeze(0).to(device)

  with torch.no_grad():
    output = model(input_tensor)
    # Get the predicted word index
    predicted_index = torch.argmax(output, dim=1).item()

  # Invert the vocab dictionary to map indices back to words
  idx_to_word = {idx: word for word, idx in vocab.items()}

  predicted_word = idx_to_word.get(predicted_index, '<unk>')
  return predicted_word


In [63]:
predict_next_word(model, "what is a", vocab, max_len = 40)

'to'

In [64]:
predict_next_word(model, "what is", vocab, max_len = 1)

'termed'

Let's test the prediction function with an example. We will use the `max_len` from our training data preparation.

In [65]:
# We need the max_len calculated during data preparation
max_len = max(len_list) # This variable was generated earlier

# Example usage:
input_text = "what is"
next_word = predict_next_word(model, input_text, vocab, max_len)
print(f"Input: '{input_text}'")
print(f"Predicted next word: '{next_word}'")

input_text = "how to avoid"
next_word = predict_next_word(model, input_text, vocab, max_len)
print(f"Input: '{input_text}'")
print(f"Predicted next word: '{next_word}'")

input_text = "what is training"
next_word = predict_next_word(model, input_text, vocab, max_len)
print(f"Input: '{input_text}'")
print(f"Predicted next word: '{next_word}'")

input_text = "what is a neural"
next_word = predict_next_word(model, input_text, vocab, max_len)
print(f"Input: '{input_text}'")
print(f"Predicted next word: '{next_word}'")

input_text = "what is cross"
next_word = predict_next_word(model, input_text, vocab, max_len)
print(f"Input: '{input_text}'")
print(f"Predicted next word: '{next_word}'")

Input: 'what is'
Predicted next word: 'centuries'
Input: 'how to avoid'
Predicted next word: 'centuries'
Input: 'what is training'
Predicted next word: ','
Input: 'what is a neural'
Predicted next word: 'centuries'
Input: 'what is cross'
Predicted next word: 'centuries'


In [69]:
import time
import builtins # Import the builtins module

num_tokens = 10
input_text = builtins.input("Enter the sentence:") # Use builtins.input
for i in range(num_tokens):
  # Corrected argument order and max_len
  output_text = predict_next_word(model, input_text, vocab, max_len)
  print(output_text)
  # To generate a sequence, we should append the predicted word to the input_text
  # and use that as the input for the next prediction.
  input_text = input_text + " " + output_text
  time.sleep(0.5)

Enter the sentence:what is gita
centuries
,
the
bhagavad
gita
has
been
considered
animportant
spiritual
